# 🦆 Member 4 — Load: DuckDB Analytical Database
**Business Goal:** Load all transformed data into DuckDB and create analytical views for the dashboard.

**Pipeline Role:** `data/processed/*.parquet` → DuckDB tables → analytical views

In [ ]:
import duckdb
import pandas as pd
import os
from datetime import datetime
from pathlib import Path

print('✅ Libraries loaded')
print(f'📅 Run time: {datetime.now()}')
print(f'🦆 DuckDB version: {duckdb.__version__}')

✅ Libraries loaded
📅 Run time: 2026-05-12 12:22:32.320281
🦆 DuckDB version: 1.5.2


In [ ]:
# ── CONNECT TO DUCKDB ─────────────────────────────────────────────
root_path = Path.cwd().resolve()
for parent in [root_path] + list(root_path.parents):
    if (parent / 'data' / 'processed').exists():
        root_path = parent
        break
    if (parent / 'notebooks' / 'data' / 'processed').exists():
        root_path = parent / 'notebooks'
        break

DATA_DIR = root_path / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
DB_PATH = DATA_DIR / 'pipeline_analytics.duckdb'

countries_file = PROCESSED_DIR / 'countries_transformed.parquet'
fx_rates_file = PROCESSED_DIR / 'fx_rates_transformed.parquet'
taxi_file = PROCESSED_DIR / 'taxi_transformed.parquet'
taxi_hourly_file = PROCESSED_DIR / 'taxi_hourly_summary.parquet'

required_files = [countries_file, fx_rates_file, taxi_file, taxi_hourly_file]
missing = [str(f) for f in required_files if not f.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing processed source files: {missing}. "
        "Run the transform/orchestration notebook first."
    )

DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(DB_PATH))
print(f'✅ Connected to DuckDB: {DB_PATH}')

✅ Connected to DuckDB: data/pipeline_analytics.duckdb


In [9]:
# ── DROP EXISTING TABLES (clean reload) ───────────────────────────
tables_to_drop = ['countries', 'fx_rates', 'taxi_trips', 'taxi_hourly']
views_to_drop = ['vw_market_overview', 'vw_profitability', 'vw_regional_revenue']
for t in tables_to_drop:
    con.execute(f'DROP TABLE IF EXISTS {t}')
for v in views_to_drop:
    con.execute(f'DROP VIEW IF EXISTS {v}')

print('✅ Cleaned existing tables')

✅ Cleaned existing tables


## 📥 Load Tables from Parquet

In [ ]:
# ── LOAD: Countries Table ─────────────────────────────────────────
con.execute(f"""
    CREATE TABLE countries AS
    SELECT * FROM read_parquet('{countries_file.as_posix()}')
""")

count = con.execute('SELECT COUNT(*) FROM countries').fetchone()[0]
print(f'✅ Table: countries — {count:,} rows')
con.execute('SELECT country_name, region, population_density, market_score, expansion_recommendation FROM countries LIMIT 5').df()

✅ Table: countries — 245 rows


,country_name,region,population_density,market_score,expansion_recommendation
0,Anguilla,Americas,175.93,6.15,MONITOR
1,Guatemala,Americas,166.04,8.24,EXPAND_NOW
2,Gambia,Africa,226.65,6.56,MONITOR
3,Mexico,Americas,66.47,7.39,MONITOR
4,Malawi,Africa,175.00,6.30,MONITOR


In [ ]:
# ── LOAD: FX Rates Table ──────────────────────────────────────────
con.execute(f"""
    CREATE TABLE fx_rates AS
    SELECT * FROM read_parquet('{fx_rates_file.as_posix()}')
""")

count = con.execute('SELECT COUNT(*) FROM fx_rates').fetchone()[0]
print(f'✅ Table: fx_rates — {count:,} rows')
con.execute("""
    SELECT currency_code, rate_to_usd, local_to_usd_rate, currency_strength
    FROM fx_rates
    WHERE currency_code IN ('ETB','EUR','GBP','JPY','CNY','NGN')
""").df()

✅ Table: fx_rates — 172 rows


,currency_code,rate_to_usd,local_to_usd_rate,currency_strength
0,CNY,6.792100,0.147230,MODERATE
1,ETB,157.000000,0.006369,VERY_WEAK
2,EUR,0.851752,1.174051,STRONG
3,GBP,0.738993,1.353193,STRONG
4,JPY,157.593500,0.006345,VERY_WEAK
5,NGN,1371.890000,0.000729,VERY_WEAK


In [ ]:
# ── LOAD: Taxi Trips Table ────────────────────────────────────────
con.execute(f"""
    CREATE TABLE taxi_trips AS
    SELECT * FROM read_parquet('{taxi_file.as_posix()}')
""")

count = con.execute('SELECT COUNT(*) FROM taxi_trips').fetchone()[0]
print(f'✅ Table: taxi_trips — {count:,} rows')

# ── LOAD: Taxi Hourly Summary ─────────────────────────────────────
con.execute(f"""
    CREATE TABLE taxi_hourly AS
    SELECT * FROM read_parquet('{taxi_hourly_file.as_posix()}')
""")

count2 = con.execute('SELECT COUNT(*) FROM taxi_hourly').fetchone()[0]
print(f'✅ Table: taxi_hourly — {count2} rows')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table: taxi_trips — 2,869,637 rows
✅ Table: taxi_hourly — 24 rows


## 🏗️ Create Analytical Views

In [ ]:
# ── VIEW 1: Market Overview ───────────────────────────────────────
con.execute("""
    CREATE VIEW vw_market_overview AS
    SELECT
        c.region,
        COUNT(*)                                    AS num_countries,
        SUM(c.population)                           AS total_population,
        ROUND(AVG(c.population_density), 2)         AS avg_density,
        ROUND(AVG(c.market_score), 2)               AS avg_market_score,
        COUNT(CASE WHEN c.expansion_recommendation = 'EXPAND_NOW' THEN 1 END) AS expand_now_count,
        COUNT(CASE WHEN c.currency_strength IN ('STRONG','MODERATE') THEN 1 END) AS stable_currency_count
    FROM countries c
    GROUP BY c.region
""")
print('✅ View: vw_market_overview')
con.execute('SELECT * FROM vw_market_overview ORDER BY avg_market_score DESC').df()

CatalogException: Catalog Error: Table with name countries does not exist!
Did you mean "main.countries"?

LINE 11:     FROM countries c
                  ^

: 

In [ ]:
# ── VIEW 2: Trip Profitability ────────────────────────────────────
con.execute("""
    CREATE VIEW vw_profitability AS
    SELECT
        time_period,
        pickup_hour,
        COUNT(*)                                           AS total_trips,
        ROUND(AVG(fare_amount), 2)                         AS avg_fare_usd,
        ROUND(AVG(revenue_per_mile), 2)                    AS avg_rev_per_mile,
        ROUND(SUM(fare_amount), 2)                         AS total_revenue_usd,
        ROUND(SUM(tip_amount), 2)                          AS total_tips_usd,
        COUNT(CASE WHEN is_profitable = true THEN 1 END)   AS profitable_trips,
        ROUND(
            COUNT(CASE WHEN is_profitable = true THEN 1 END) * 100.0 / COUNT(*),
            1
        )                                                  AS profitable_pct
    FROM taxi_trips
    GROUP BY time_period, pickup_hour
""")
print('✅ View: vw_profitability')
con.execute('SELECT * FROM vw_profitability ORDER BY total_revenue_usd DESC LIMIT 10').df()

✅ View: vw_profitability


,time_period,pickup_hour,total_trips,avg_fare_usd,avg_rev_per_mile,total_revenue_usd,total_tips_usd,profitable_trips,profitable_pct
0,EVENING_RUSH,17,200284,18.12,11.31,3628930.33,712274.67,197349,98.5
1,OFF_PEAK,16,184952,19.46,12.69,3598526.72,691421.43,182275,98.6
2,OFF_PEAK,15,183973,19.11,12.45,3516049.41,625032.90,181324,98.6
3,EVENING_RUSH,18,206263,17.01,10.92,3509508.47,706677.82,203234,98.5
4,LUNCH_HOUR,14,178006,19.27,12.49,3430498.41,614392.12,175422,98.5
5,EVENING_RUSH,19,178787,17.63,11.26,3151502.12,629906.25,176021,98.5
6,LUNCH_HOUR,13,165333,18.42,11.78,3045770.17,544655.08,162771,98.5
7,OFF_PEAK,21,155890,18.29,10.04,2851661.55,559356.93,153795,98.7
8,LUNCH_HOUR,12,159891,17.80,11.71,2845688.48,509643.30,157465,98.5
9,OFF_PEAK,20,155544,18.05,11.05,2808068.54,551925.44,153262,98.5


In [ ]:
# ── VIEW 3: Regional Revenue with Currency Impact ─────────────────
con.execute("""
    CREATE VIEW vw_regional_revenue AS
    SELECT
        c.region,
        c.country_name,
        c.currency_code,
        c.currency_strength,
        c.population,
        c.population_density,
        c.market_score,
        c.expansion_recommendation,
        f.rate_to_usd,
        f.local_to_usd_rate,
        -- Simulated revenue if avg NYC fare applied to market
        ROUND(15.50 * f.rate_to_usd, 2)             AS avg_fare_in_local_currency,
        ROUND(c.population * 0.001 * 15.50, 0)      AS estimated_annual_revenue_usd
    FROM countries c
    LEFT JOIN fx_rates f ON c.currency_code = f.currency_code
""")
print('✅ View: vw_regional_revenue')
con.execute('SELECT * FROM vw_regional_revenue ORDER BY estimated_annual_revenue_usd DESC LIMIT 10').df()

✅ View: vw_regional_revenue


,region,country_name,currency_code,currency_strength,population,population_density,market_score,expansion_recommendation,rate_to_usd,local_to_usd_rate,avg_fare_in_local_currency,estimated_annual_revenue_usd
0,Asia,India,INR,WEAK,1417492000,431.21,8.83,EXPAND_NOW,95.342843,0.010488,1477.81,21971126.0
1,Asia,China,CNY,MODERATE,1408280000,145.08,9.50,EXPAND_NOW,6.795000,0.147167,105.32,21828340.0
2,Americas,United States,USD,STRONG,340110988,35.71,10.00,EXPAND_NOW,1.000000,1.000000,15.50,5271720.0
3,Asia,Indonesia,IDR,VERY_WEAK,284438782,149.35,7.03,MONITOR,17420.343274,0.000057,270015.32,4408801.0
4,Asia,Pakistan,PKR,VERY_WEAK,241499431,303.36,7.29,MONITOR,278.600000,0.003589,4318.30,3743241.0
5,Africa,Nigeria,NGN,VERY_WEAK,223800000,242.27,7.15,MONITOR,1367.400000,0.000731,21194.70,3468900.0
6,Americas,Brazil,BRL,MODERATE,213421037,25.06,8.63,EXPAND_NOW,4.910000,0.203666,76.11,3308026.0
7,Asia,Bangladesh,BDT,VERY_WEAK,169828911,1150.84,8.09,EXPAND_NOW,122.936124,0.008134,1905.51,2632348.0
8,Europe,Russia,RUB,WEAK,146028325,8.54,7.27,MONITOR,73.600925,0.013587,1140.81,2263439.0
9,Americas,Mexico,MXN,WEAK,130575786,66.47,7.39,MONITOR,17.193843,0.058160,266.50,2023925.0


## 🔍 Analytical Queries

In [10]:
# ── KEY INSIGHT QUERIES ───────────────────────────────────────────
print('=' * 60)
print('📊 KEY BUSINESS INSIGHTS FROM DUCKDB')
print('=' * 60)

print('\n🔍 INSIGHT 1: Most Profitable Hours')
con.execute("""
    SELECT pickup_hour, time_period, total_revenue_usd, profitable_pct
    FROM vw_profitability
    ORDER BY total_revenue_usd DESC
    LIMIT 5
""").df()

📊 KEY BUSINESS INSIGHTS FROM DUCKDB

🔍 INSIGHT 1: Most Profitable Hours


,pickup_hour,time_period,total_revenue_usd,profitable_pct
0,17,EVENING_RUSH,3628930.33,98.5
1,16,OFF_PEAK,3598526.72,98.6
2,15,OFF_PEAK,3516049.41,98.6
3,18,EVENING_RUSH,3509508.47,98.5
4,14,LUNCH_HOUR,3430498.41,98.5


In [11]:
print('\n🔍 INSIGHT 2: Best Markets for Expansion')
con.execute("""
    SELECT country_name, region, market_score,
           expansion_recommendation, currency_strength
    FROM vw_regional_revenue
    WHERE expansion_recommendation = 'EXPAND_NOW'
    ORDER BY market_score DESC
    LIMIT 10
""").df()


🔍 INSIGHT 2: Best Markets for Expansion


,country_name,region,market_score,expansion_recommendation,currency_strength
0,Bahrain,Asia,10.31,EXPAND_NOW,STRONG
1,Singapore,Asia,10.17,EXPAND_NOW,MODERATE
2,Hong Kong,Asia,10.16,EXPAND_NOW,MODERATE
3,United Kingdom,Europe,10.08,EXPAND_NOW,STRONG
4,Germany,Europe,10.04,EXPAND_NOW,STRONG
5,Monaco,Europe,10.02,EXPAND_NOW,STRONG
6,United States,Americas,10.00,EXPAND_NOW,STRONG
7,Netherlands,Europe,9.92,EXPAND_NOW,STRONG
8,Malta,Europe,9.91,EXPAND_NOW,STRONG
9,Italy,Europe,9.86,EXPAND_NOW,STRONG


In [12]:
print('\n🔍 INSIGHT 3: Unprofitable Trip Hours (Action Required)')
con.execute("""
    SELECT pickup_hour, time_period, total_trips,
           avg_fare_usd, profitable_pct,
           ROUND(100 - profitable_pct, 1) AS unprofitable_pct
    FROM vw_profitability
    WHERE profitable_pct < 80
    ORDER BY unprofitable_pct DESC
""").df()


🔍 INSIGHT 3: Unprofitable Trip Hours (Action Required)


,pickup_hour,time_period,total_trips,avg_fare_usd,profitable_pct,unprofitable_pct


In [13]:
# ── DATABASE SUMMARY ──────────────────────────────────────────────
print('\n📦 DATABASE SUMMARY:')
tables = con.execute("SHOW TABLES").df()
print(tables.to_string())

for table in ['countries', 'fx_rates', 'taxi_trips', 'taxi_hourly']:
    try:
        cnt = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
        print(f'  📊 {table}: {cnt:,} rows')
    except:
        pass

con.close()
size_mb = os.path.getsize(DB_PATH) / (1024*1024)
print(f'\n💾 DuckDB file size: {size_mb:.1f} MB')
print(f'📍 Location: {DB_PATH}')
print('\n🏁 Member 4 COMPLETE — DuckDB ready for Member 5 (Dashboard)')


📦 DATABASE SUMMARY:
                  name
0            countries
1             fx_rates
2          taxi_hourly
3           taxi_trips
4   vw_market_overview
5     vw_profitability
6  vw_regional_revenue
  📊 countries: 245 rows
  📊 fx_rates: 172 rows
  📊 taxi_trips: 2,869,637 rows
  📊 taxi_hourly: 24 rows

💾 DuckDB file size: 79.5 MB
📍 Location: data/pipeline_analytics.duckdb

🏁 Member 4 COMPLETE — DuckDB ready for Member 5 (Dashboard)
